# 🚀 Stage 04: Deployment & Analytics Automation
Project: Brazilian E-commerce Sales Performance & Growth Analytics

👤 Role & Responsibility
Role: Senior Marketing Data Analyst (MDA) & Analytics Engineer.

Responsibility: Mentransformasikan model prediktif statis menjadi sistem otomasi yang mampu menghasilkan proyeksi pertumbuhan secara real-time untuk mendukung pengambilan keputusan strategis.

📊 Dataset Reference
Source: Olist Brazilian E-commerce Public Dataset.

Target File: 01_olist_master_join_cleaned.csv (Gold Layer).

Artifacts: growth_predictive_model.pkl & feature_scaler.pkl.

🎯 Project Focus
Operational Excellence: Otomatisasi Inference Pipeline untuk menghilangkan proses pengolahan data manual.

Growth Mechanics: Pemantauan berkelanjutan terhadap Freight-to-Price Ratio dan dampaknya terhadap volume pesanan.

Scalability: Membangun infrastruktur kode yang modular untuk memproses data transaksi baru dalam skala besar.

🏁 Project Goal
Automated Insights: Menghasilkan laporan mingguan otomatis mengenai potensi kenaikan GMV (Revenue Upside).

Prescriptive Actions: Memberikan rekomendasi prioritas kategori produk untuk intervensi logistik secara dinamis.

Business Impact: Memastikan perusahaan memiliki visibilitas penuh terhadap peluang pertumbuhan yang belum tergarap (untapped growth) berbasis model Machine Learning.

In [ ]:
import pandas as pd
import joblib
import os
from datetime import datetime
# 1. DEFINISI PATH (Gunakan path relatif yang lebih aman)
# Menyesuaikan dengan instruksi struktur: ../../outputs/models/
BASE_DIR = os.getcwd()
MODEL_PATH = os.path.join(BASE_DIR, '../../outputs/models/growth_predictive_model.pkl')
SCALER_PATH = os.path.join(BASE_DIR, '../../outputs/models/feature_scaler.pkl')

def load_assets():
    """Memuat model dan scaler secara aman dengan validasi keberadaan file."""
    
    # Validasi File secara eksplisit untuk menghindari FileNotFoundError
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"❌ Critical Error: File Model tidak ditemukan di {os.path.abspath(MODEL_PATH)}")
        
    if not os.path.exists(SCALER_PATH):
        raise FileNotFoundError(f"❌ Critical Error: File Scaler tidak ditemukan di {os.path.abspath(SCALER_PATH)}")
    
    try:
        # Memuat file menggunakan joblib (Standard Gold for Random Forest)
        model = joblib.load(MODEL_PATH)
        scaler = joblib.load(SCALER_PATH)
        
        print("✅ Assets Loaded: Model dan Scaler siap digunakan untuk Sales Performance Analytics.")
        return model, scaler
        
    except Exception as e:
        print(f"❌ Error saat memuat file pkl: {str(e)}")
        return None, None

# --- Eksekusi Inisialisasi ---
model, scaler = load_assets()

✅ Assets Loaded: Model dan Scaler siap digunakan untuk Sales Performance Analytics.


In [ ]:
def preprocess_automated(df_raw):
    """Transformasi data mentah menjadi fitur siap pakai untuk model."""
    if df_raw.empty:
        return None, None
    
    df = df_raw.copy()
    
    # 1. Feature Engineering: Freight & Friction
    df['freight_ratio'] = df['freight_value'] / df['price']
    df['is_high_friction'] = (df['freight_ratio'] > 0.2).astype(int)
    
    # 2. Regional Engineering (Sesuai Diagnostic image_40eb36.png)
    north_states = ['AM', 'RR', 'AP', 'PA', 'AC', 'RO', 'TO']
    df['is_north_region'] = df['customer_state'].apply(lambda x: 1 if x in north_states else 0)
    
    # 3. One-Hot Encoding untuk kategorikal
    # Catatan: Kita tidak khawatir jika jumlah kolom berbeda, akan diperbaiki di Section 3
    df_encoded = pd.get_dummies(df, columns=['customer_state', 'product_category_name_english'])
    
    # Seleksi fitur numerik saja
    X_processed = df_encoded.select_dtypes(include=['number'])
    
    return df, X_processed

In [ ]:
def run_production_inference(model, scaler, df_raw, X_input):
    try:
        # SOLUSI PERMANEN: Menyelaraskan 102 kolom yang diminta model
        # Ini akan menambah kolom dummy yang hilang secara otomatis
        expected_cols = model.feature_names_in_
        X_aligned = X_input.reindex(columns=expected_cols, fill_value=0)
        
        # Scaling & Prediction
        X_scaled = scaler.transform(X_aligned)
        df_res = df_raw.copy()
        df_res['predicted_volume'] = model.predict(X_scaled)
        
        # Kalkulasi Growth Metrics
        df_res['revenue_upside'] = (df_res['predicted_volume'] * df_res['price']) - df_res['price']
        
        return df_res
    except Exception as e:
        print(f"❌ Automation Error: {str(e)}")
        return None

In [ ]:
# =================================================================
# 5. Scheduled Report Export (PROFESSIONAL & ERROR-FREE VERSION)
# =================================================================
import os
import pandas as pd
from datetime import datetime

def generate_automated_report(df_input, output_folder):
    """
    Fungsi untuk mengekspor laporan pertumbuhan sales secara otomatis.
    Didesain agar tahan banting terhadap error 'NoneType' atau data kosong.
    """
    print(f"🕒 Memulai proses export pada: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # 1. GATEKEEPER: Validasi Input (Mencegah error image_3dc3e3.png)
    if df_input is None:
        print("❌ ABORTED: Data input bernilai 'None'. Periksa kembali Langkah 4 (Inference).")
        return None
    
    if not isinstance(df_input, pd.DataFrame) or df_input.empty:
        print("⚠️ SKIPPED: Data tersedia tapi kosong. Tidak ada yang bisa di-export.")
        return None

    try:
        # 2. PREPARASI DIREKTORI
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)
            print(f"📁 Folder baru dibuat: {output_folder}")

        # 3. LOGIKA AGREGASI (Sales Performance & Growth Focus)
        # Mengelompokkan berdasarkan State & Kategori untuk melihat 'Revenue Upside'
        report_summary = df_input.groupby(['customer_state', 'product_category_name_english']).agg({
            'price': 'mean',
            'freight_value': 'mean',
            'predicted_volume': 'sum',
            'revenue_upside': 'sum'
        }).reset_index()

        # Menambahkan kolom metadata untuk Audit Trail
        report_summary['report_generated_at'] = datetime.now().strftime('%Y-%m-%d %H:%M')

        # 4. EXPORT DENGAN TIMESTAMP (Dinamis)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        file_name = f"olist_growth_analytics_{timestamp}.csv"
        full_path = os.path.join(output_folder, file_name)
        
        report_summary.to_csv(full_path, index=False)
        
        print(f"✅ SUCCESS: Laporan berhasil disimpan di: {full_path}")
        print(f"📊 Total Potensi Pertumbuhan: BRL {report_summary['revenue_upside'].sum():,.2f}")
        
        return report_summary

    except Exception as e:
        # Menangkap error tak terduga tanpa menghentikan seluruh notebook
        print(f"❌ CRITICAL ERROR di Cell 5: {str(e)}")
        return None

# --- EKSEKUSI CELL 5 ---
# Memastikan OUTPUT_DIR sudah didefinisikan sebelumnya
final_report = generate_automated_report(df_final, '../../outputs/reports/')

🕒 Memulai proses export pada: 2026-01-27 21:36:51
⚠️ SKIPPED: Data tersedia tapi kosong. Tidak ada yang bisa di-export.
